In [ ]:
# Semantic Segmentation with ResNet on Pascal VOC Dataset
# Refined implementation with improved architecture and training process

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as F
from torch.utils.data import DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os
import time
from datetime import datetime
from typing import Tuple, List, Dict, Optional

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
if device.type == 'cuda':
    torch.cuda.manual_seed(42)
np.random.seed(42)

# =============================================================================
# 1. Enhanced Dataset Loading and Preprocessing
# =============================================================================

class VOCSegmentationWithAugmentation(torchvision.datasets.VOCSegmentation):
    """Extended VOC dataset with additional augmentations"""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        
    def __getitem__(self, index):
        image, target = super().__getitem__(index)
        
        # Apply additional augmentations for training
        if self.image_set == 'train':
            # Random horizontal flipping
            if np.random.random() > 0.5:
                image = F.hflip(image)
                target = F.hflip(target)
            
            # Random slight rotation (-10 to 10 degrees)
            angle = np.random.uniform(-10, 10)
            image = F.rotate(image, angle)
            target = F.rotate(target, angle, interpolation=Image.NEAREST)
            
            # Random brightness and contrast adjustments
            brightness_factor = np.random.uniform(0.9, 1.1)
            contrast_factor = np.random.uniform(0.9, 1.1)
            image = F.adjust_brightness(image, brightness_factor)
            image = F.adjust_contrast(image, contrast_factor)
        
        return image, target

# Define transforms with separate normalization for images and targets
image_transform = transforms.Compose([
    transforms.Resize((256, 256)),  # Slightly larger for random cropping
    transforms.CenterCrop(224),     # RandomCrop for training in actual implementation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

target_transform = transforms.Compose([
    transforms.Resize((256, 256), interpolation=Image.NEAREST),
    transforms.CenterCrop(224),  # Match the image crop
    transforms.PILToTensor()     # Keeps as integer tensor without normalization
])

# Load datasets
print("Loading Pascal VOC dataset...")
train_dataset = VOCSegmentationWithAugmentation(
    root='./data',
    year='2012',
    image_set='train',
    download=True,
    transform=image_transform,
    target_transform=target_transform
)

val_dataset = torchvision.datasets.VOCSegmentation(
    root='./data',
    year='2012',
    image_set='val',
    download=True,
    transform=image_transform,
    target_transform=target_transform
)

# Create data loaders with appropriate settings
batch_size = 16 if device.type == 'cuda' else 4
num_workers = 4 if device.type == 'cuda' else 2

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True, 
    num_workers=num_workers,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=num_workers,
    pin_memory=True
)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Batch size: {batch_size}")

# Pascal VOC class names (21 classes including background)
class_names = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus',
    'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike',
    'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

# Color map for visualization
def create_colormap():
    colormap = np.zeros((256, 3), dtype=np.uint8)
    colors = [
        [0, 0, 0], [128, 0, 0], [0, 128, 0], [128, 128, 0], [0, 0, 128],
        [128, 0, 128], [0, 128, 128], [128, 128, 128], [64, 0, 0], [192, 0, 0],
        [64, 128, 0], [192, 128, 0], [64, 0, 128], [192, 0, 128], [64, 128, 128],
        [192, 128, 128], [0, 64, 0], [128, 64, 0], [0, 192, 0], [128, 192, 0],
        [0, 64, 128]
    ]
    for i, color in enumerate(colors):
        colormap[i] = color
    # Set void pixels (255) to white for visualization
    colormap[255] = [255, 255, 255]
    return colormap

colormap = create_colormap()

def visualize_sample(dataset, idx=0):
    """Visualize a sample from the dataset"""
    image, target = dataset[idx]

    # Denormalize image for visualization
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    image_denorm = image * std + mean
    image_denorm = torch.clamp(image_denorm, 0, 1)

    # Convert target to numpy (already integer tensor)
    target_np = target.squeeze(0).numpy().astype(np.uint8)
    target_colored = colormap[target_np]

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.imshow(image_denorm.permute(1, 2, 0))
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(target_colored)
    plt.title('Segmentation Mask')
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# Visualize some samples
print("\nVisualizing sample images and their segmentation masks:")
for i in range(2):
    visualize_sample(train_dataset, i)

# =============================================================================
# 2. Improved Model Definition with Skip Connections
# =============================================================================

class ResNetSegmentation(nn.Module):
    def __init__(self, num_classes=21, backbone='resnet34', pretrained=True):
        super(ResNetSegmentation, self).__init__()
        
        # Load pretrained ResNet
        if backbone == 'resnet18':
            resnet = torchvision.models.resnet18(pretrained=pretrained)
            expansion = 1
            features = 512
        elif backbone == 'resnet34':
            resnet = torchvision.models.resnet34(pretrained=pretrained)
            expansion = 1
            features = 512
        elif backbone == 'resnet50':
            resnet = torchvision.models.resnet50(pretrained=pretrained)
            expansion = 4
            features = 2048
        else:
            raise ValueError(f"Unsupported backbone: {backbone}")

        # Early layers
        self.initial = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool
        )
        
        # Encoder with skip connections
        self.layer1 = resnet.layer1  # 1/4
        self.layer2 = resnet.layer2  # 1/8
        self.layer3 = resnet.layer3  # 1/16
        self.layer4 = resnet.layer4  # 1/32
        
        # Decoder with skip connections
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        
        # Decoder blocks
        self.decoder4 = self._make_decoder_block(features, features // 2)
        self.decoder3 = self._make_decoder_block(features // 2, features // 4)
        self.decoder2 = self._make_decoder_block(features // 4, features // 8)
        self.decoder1 = self._make_decoder_block(features // 8, features // 16)
        
        # Final convolution
        self.final = nn.Sequential(
            nn.Conv2d(features // 16, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, num_classes, kernel_size=1)
        )
        
    def _make_decoder_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Encoder
        x0 = self.initial(x)      # 1/4
        x1 = self.layer1(x0)      # 1/4
        x2 = self.layer2(x1)      # 1/8
        x3 = self.layer3(x2)      # 1/16
        x4 = self.layer4(x3)      # 1/32
        
        # Decoder with skip connections
        d4 = self.upsample(x4)    # 1/16
        d4 = torch.cat([d4, x3], dim=1)
        d4 = self.decoder4(d4)
        
        d3 = self.upsample(d4)    # 1/8
        d3 = torch.cat([d3, x2], dim=1)
        d3 = self.decoder3(d3)
        
        d2 = self.upsample(d3)    # 1/4
        d2 = torch.cat([d2, x1], dim=1)
        d2 = self.decoder2(d2)
        
        d1 = self.upsample(d2)    # 1/2
        d1 = self.decoder1(d1)
        
        # Final upsampling and convolution
        out = self.upsample(d1)   # Original size
        out = self.final(out)
        
        return out

# Create model
model = ResNetSegmentation(num_classes=21, backbone='resnet34', pretrained=True).to(device)
print(f"\nModel created and moved to {device}")

# Print model summary
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total trainable parameters: {count_parameters(model):,}")

# =============================================================================
# 3. Enhanced Training Setup
# =============================================================================

# Loss function with class weighting (optional)
# You could calculate class weights based on dataset statistics
criterion = nn.CrossEntropyLoss(ignore_index=255)  # 255 is typically used for void/ignore pixels

# Optimizer with different learning rates for backbone and decoder
backbone_params = []
decoder_params = []
for name, param in model.named_parameters():
    if 'initial' in name or 'layer' in name:
        backbone_params.append(param)
    else:
        decoder_params.append(param)

optimizer = optim.Adam([
    {'params': backbone_params, 'lr': 0.0001},
    {'params': decoder_params, 'lr': 0.001}
], weight_decay=1e-4)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5, verbose=True
)

# Initialize TensorBoard for visualization
log_dir = f"logs/{datetime.now().strftime('%Y%m%d-%H%M%S')}"
writer = SummaryWriter(log_dir)

# Training function with timing and gradient clipping
def train_epoch(model, dataloader, criterion, optimizer, device, epoch, grad_clip=1.0):
    model.train()
    running_loss = 0.0
    correct_pixels = 0
    total_pixels = 0
    start_time = time.time()
    
    for i, (images, targets) in enumerate(dataloader):
        images = images.to(device, non_blocking=True)
        targets = targets.squeeze(1).long().to(device, non_blocking=True)  # Remove channel dimension

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        
        # Gradient clipping
        if grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            
        optimizer.step()

        running_loss += loss.item()
        
        # Calculate pixel accuracy
        predictions = torch.argmax(outputs, dim=1)
        mask = targets != 255  # Ignore void pixels
        correct_pixels += (predictions[mask] == targets[mask]).sum().item()
        total_pixels += mask.sum().item()

        if i % 50 == 0:
            batch_time = time.time() - start_time
            samples_per_sec = (i + 1) * batch_size / batch_time
            print(f'Epoch {epoch}, Batch {i}/{len(dataloader)}, Loss: {loss.item():.4f}, '
                  f'Samples/s: {samples_per_sec:.1f}')

    epoch_loss = running_loss / len(dataloader)
    epoch_accuracy = correct_pixels / total_pixels if total_pixels > 0 else 0
    
    return epoch_loss, epoch_accuracy

# Enhanced validation function with mIoU calculation
def validate_epoch(model, dataloader, criterion, device, num_classes=21):
    model.eval()
    running_loss = 0.0
    correct_pixels = 0
    total_pixels = 0
    
    # For mIoU calculation
    confusion_matrix = np.zeros((num_classes, num_classes), dtype=np.int64)
    
    with torch.no_grad():
        for images, targets in dataloader:
            images = images.to(device, non_blocking=True)
            targets = targets.squeeze(1).long().to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, targets)
            running_loss += loss.item()

            # Calculate pixel accuracy
            predictions = torch.argmax(outputs, dim=1)
            mask = targets != 255  # Ignore void pixels
            correct_pixels += (predictions[mask] == targets[mask]).sum().item()
            total_pixels += mask.sum().item()
            
            # Update confusion matrix for mIoU
            for t, p in zip(targets.view(-1), predictions.view(-1)):
                if t < num_classes:  # Skip void pixels
                    confusion_matrix[t.long(), p.long()] += 1

    # Calculate mIoU
    iou_per_class = np.zeros(num_classes)
    for i in range(num_classes):
        tp = confusion_matrix[i, i]
        fp = np.sum(confusion_matrix[:, i]) - tp
        fn = np.sum(confusion_matrix[i, :]) - tp
        iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
        iou_per_class[i] = iou
    
    mean_iou = np.nanmean(iou_per_class)
    
    epoch_loss = running_loss / len(dataloader)
    epoch_accuracy = correct_pixels / total_pixels if total_pixels > 0 else 0
    
    return epoch_loss, epoch_accuracy, mean_iou, iou_per_class

# =============================================================================
# 4. Enhanced Training Loop with Early Stopping
# =============================================================================

num_epochs = 50
train_losses = []
val_losses = []
val_accuracies = []
val_ious = []
best_iou = 0.0
patience = 10
patience_counter = 0

print(f"\nStarting training for {num_epochs} epochs...")

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')
    print('-' * 50)

    # Training
    epoch_start = time.time()
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, epoch+1)
    train_time = time.time() - epoch_start
    
    # Validation
    val_start = time.time()
    val_loss, val_acc, val_iou, iou_per_class = validate_epoch(model, val_loader, criterion, device)
    val_time = time.time() - val_start

    # Update learning rate
    scheduler.step(val_iou)
    
    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    val_ious.append(val_iou)
    
    # Log to TensorBoard
    writer.add_scalar('Loss/Train', train_loss, epoch)
    writer.add_scalar('Loss/Validation', val_loss, epoch)
    writer.add_scalar('Accuracy/Train', train_acc, epoch)
    writer.add_scalar('Accuracy/Validation', val_acc, epoch)
    writer.add_scalar('mIoU/Validation', val_iou, epoch)
    writer.add_scalar('Learning Rate', optimizer.param_groups[0]['lr'], epoch)
    
    # Print metrics
    print(f'Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}')
    print(f'Val Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}, mIoU: {val_iou:.4f}')
    print(f'Epoch Time: {train_time + val_time:.2f}s (Train: {train_time:.2f}s, Val: {val_time:.2f}s)')
    
    # Save best model
    if val_iou > best_iou:
        best_iou = val_iou
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_losses': train_losses,
            'val_losses': val_losses,
            'val_accuracies': val_accuracies,
            'val_ious': val_ious,
            'best_iou': best_iou,
        }, 'best_model.pth')
        print(f"Saved new best model with mIoU: {best_iou:.4f}")
    else:
        patience_counter += 1
        print(f"No improvement in mIoU. Patience: {patience_counter}/{patience}")
    
    # Early stopping
    if patience_counter >= patience:
        print(f"Early stopping triggered after {epoch+1} epochs")
        break

print("\nTraining completed!")
writer.close()

# =============================================================================
# 5. Enhanced Results Visualization
# =============================================================================

# Load best model
checkpoint = torch.load('best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']} with mIoU: {checkpoint['best_iou']:.4f}")

# Plot training curves
plt.figure(figsize=(15, 10))

plt.subplot(2, 2, 1)
plt.plot(range(1, len(train_losses)+1), train_losses, 'b-', label='Training Loss')
plt.plot(range(1, len(val_losses)+1), val_losses, 'r-', label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

plt.subplot(2, 2, 2)
plt.plot(range(1, len(val_accuracies)+1), val_accuracies, 'g-', label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Validation Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(2, 2, 3)
plt.plot(range(1, len(val_ious)+1), val_ious, 'm-', label='Validation mIoU')
plt.xlabel('Epoch')
plt.ylabel('mIoU')
plt.title('Validation Mean IoU')
plt.legend()
plt.grid(True)

plt.subplot(2, 2, 4)
# Show class-wise IoU for the last epoch
class_iou = iou_per_class
plt.barh(range(len(class_iou)), class_iou, color='orange')
plt.yticks(range(len(class_iou)), class_names)
plt.xlabel('IoU')
plt.title('Class-wise IoU')
plt.tight_layout()

plt.savefig('training_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

# =============================================================================
# 6. Enhanced Prediction Visualization
# =============================================================================

def visualize_predictions(model, dataset, device, num_samples=5):
    """Visualize model predictions with comparison to ground truth"""
    model.eval()
    
    # Select random samples
    indices = np.random.choice(len(dataset), num_samples, replace=False)
    
    with torch.no_grad():
        for i, idx in enumerate(indices):
            image, target = dataset[idx]
            image_batch = image.unsqueeze(0).to(device)

            # Get prediction
            output = model(image_batch)
            prediction = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()

            # Denormalize image for visualization
            mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
            image_denorm = image * std + mean
            image_denorm = torch.clamp(image_denorm, 0, 1)

            # Convert masks to color
            target_np = target.squeeze(0).numpy().astype(np.uint8)
            target_colored = colormap[target_np]
            prediction_colored = colormap[prediction]
            
            # Calculate accuracy for this sample
            mask = target_np != 255
            sample_acc = np.mean((prediction[mask] == target_np[mask]).astype(np.float32))

            plt.figure(figsize=(15, 5))

            plt.subplot(1, 3, 1)
            plt.imshow(image_denorm.permute(1, 2, 0))
            plt.title('Original Image')
            plt.axis('off')

            plt.subplot(1, 3, 2)
            plt.imshow(target_colored)
            plt.title('Ground Truth')
            plt.axis('off')

            plt.subplot(1, 3, 3)
            plt.imshow(prediction_colored)
            plt.title(f'Prediction (Acc: {sample_acc:.3f})')
            plt.axis('off')

            plt.tight_layout()
            plt.savefig(f'prediction_sample_{i}.png', dpi=300, bbox_inches='tight')
            plt.show()

print("\nVisualizing predictions on validation samples:")
visualize_predictions(model, val_dataset, device, num_samples=5)

# =============================================================================
# 7. Model Evaluation and Summary
# =============================================================================

# Final evaluation on validation set
print("\nRunning final evaluation on validation set...")
val_loss, val_acc, val_iou, iou_per_class = validate_epoch(model, val_loader, criterion, device)

print(f"\n{'='*60}")
print("FINAL EVALUATION RESULTS")
print(f"{'='*60}")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")
print(f"Validation mIoU: {val_iou:.4f}")
print(f"{'='*60}")
print("Class-wise IoU:")
for i, (class_name, iou) in enumerate(zip(class_names, iou_per_class)):
    print(f"  {class_name:15}: {iou:.4f}")
print(f"{'='*60}")

# Save final model
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': class_names,
    'val_iou': val_iou,
    'val_acc': val_acc,
}, 'final_segmentation_model.pth')

print("\nFinal model saved as 'final_segmentation_model.pth'")